# Re-analysis

Re-process an experiment that is already on disk, without a microscope. Raw
images are read from the original folder and never changed; new labels, tracks
and features go to a new output folder. Fill in the cells marked **TODO** and
delete their `raise` line.

See [Re-analysis](https://github.com/pertzlab/faro/blob/main/README.md#re-analysis) in the README.

In [ ]:
import os

import pandas as pd

from faro.core.conversion import load_events_json
from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline_post import ImageProcessingPipeline_postExperiment
from faro.core.writers import OmeZarrWriter, TiffWriter

## TODO: paths

Fill in this cell, then delete the `raise` line. `SRC_PATH` holds the original experiment with its `events.json`
and either `acquisition.ome.zarr` or a `raw/` folder.

In [ ]:
SRC_PATH = r"D:\data\2026-01-01_my_experiment"
OUT_PATH = SRC_PATH + "_reanalysis"
raise NotImplementedError("TODO: set the paths")

## TODO: pipeline components

Fill in this cell, then delete the `raise` line. Set `USE_OLD_SEGMENTATIONS = True` to keep the original labels and
only re-run tracking and features; the segmentator is then ignored.

In [ ]:
from faro.segmentation.base import OtsuSegmentator
from faro.feature_extraction.simple import SimpleFE
from faro.tracking.trackpy import TrackerTrackpy

segmentators = [
    SegmentationMethod(name="labels", segmentation_class=OtsuSegmentator(), use_channel=0, save_tracked=True),
]
feature_extractor = SimpleFE("labels")
tracker = TrackerTrackpy(search_range=30)
stimulator = None                  # only needed to re-compute stimulation masks
USE_OLD_SEGMENTATIONS = False
USE_OLD_STIM_MASKS = True
raise NotImplementedError("TODO: choose the pipeline components")

## Run

`events.json` restores the acquisition plan. Each field of view runs in its
own thread; raw data is hard-linked into the output store instead of copied.

In [ ]:
events = load_events_json(SRC_PATH)

pipeline = ImageProcessingPipeline_postExperiment(
    img_storage_path=SRC_PATH,
    out_path=OUT_PATH,
    events=events,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    use_old_segmentations=USE_OLD_SEGMENTATIONS,
    use_old_stim_masks=USE_OLD_STIM_MASKS,
    n_jobs=4,
    writer=OmeZarrWriter(OUT_PATH),
)
pipeline.run()
pipeline.concat_fovs()

## Results

`exp_data.parquet` combines the tracks of all fields of view.

In [ ]:
exp_data = pd.read_parquet(os.path.join(OUT_PATH, "exp_data.parquet"))
print(f"{exp_data['particle'].nunique()} tracks, {len(exp_data)} rows")
exp_data.head()